In [1]:

import sys
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "reportlab", "pyarrow"])
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT

OUTPUT_FILE_NAME = "03 Raw Data Storage- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_ROOT: {RAW_ROOT}")
print(f"REPORTS_DIR: {REPORTS_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

# ============================================================
# 2) HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def collect_files(base_dir, patterns):
    results = []
    if not base_dir.exists():
        return results
    for pattern in patterns:
        results.extend(base_dir.rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not path.exists():
        return None
    stat = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(stat.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=40, max_chars=3500):
    if not path or not path.exists():
        return "File not found."
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        joined = "\n".join(lines)
        return joined[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def build_tree_text(base_path, max_depth=6, max_items=300):
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped_lines = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped_lines.append("")
            continue
        wrapped = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped_lines.extend(wrapped if wrapped else [""])
    return "\n".join(wrapped_lines)

def wrap_path_for_pdf(value, max_chunk=34):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(part, width=max_chunk, break_long_words=True, break_on_hyphens=True)
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=42):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(word, width=max_len, break_long_words=True, break_on_hyphens=True)
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []

    converted = []
    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table

def find_storage_scripts_and_configs():
    patterns = [
        "*upload*.py", "*store*.py", "*storage*.py", "*ingest*.py", "*collect*.py", "*load*.py",
        "*upload*.ipynb", "*store*.ipynb", "*storage*.ipynb", "*ingest*.ipynb", "*collect*.ipynb", "*load*.ipynb",
        "*.yaml", "*.yml", "*.json", "*.toml", "*.ini", "*.cfg", "*.conf", ".env", "*.properties",
    ]
    matches = []
    for base_dir in [PROJECT_ROOT, SRC_DIR]:
        if base_dir.exists():
            matches.extend(collect_files(base_dir, patterns))

    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "\\venv\\" in p_str or "/venv/" in p_str:
            continue
        if "\\.venv\\" in p_str or "/.venv/" in p_str:
            continue
        if "\\site-packages\\" in p_str or "/site-packages/" in p_str:
            continue
        cleaned.append(p)
    return cleaned

def find_logs_and_reports():
    patterns = [
        "*.log", "*.txt", "*report*.json", "*report*.csv", "*report*.pdf",
        "*summary*.csv", "*issues*.csv", "*fix*.csv"
    ]
    matches = []
    for base_dir in [REPORTS_DIR, SRC_DIR, PROJECT_ROOT]:
        if base_dir.exists():
            matches.extend(collect_files(base_dir, patterns))
    return sorted(set(matches))

def detect_storage_observations(raw_files):
    source_names = set()
    has_load_date = False
    has_load_hour = False
    extensions = set()

    for p in raw_files:
        try:
            rel = p.relative_to(RAW_ROOT)
            parts = rel.parts
            if len(parts) > 0:
                source_names.add(parts[0])
            rel_text = safe_str(rel)
            if "load_date=" in rel_text:
                has_load_date = True
            if "load_hour=" in rel_text:
                has_load_hour = True
            extensions.add(p.suffix.lower() or "[no extension]")
        except Exception:
            pass

    observations = []
    observations.append(f"Raw files discovered: {len(raw_files)}")
    observations.append(f"Sources identified: {', '.join(sorted(source_names)) if source_names else 'None'}")
    observations.append(f"Timestamp partition load_date found: {'Yes' if has_load_date else 'No'}")
    observations.append(f"Timestamp partition load_hour found: {'Yes' if has_load_hour else 'No'}")
    observations.append(f"File types found: {', '.join(sorted(extensions)) if extensions else 'None'}")
    return observations

# ============================================================
# 3) STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.4,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.3,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.2,
    leading=8.6,
    alignment=TA_LEFT,
)

# ============================================================
# 4) COLLECT PROJECT EVIDENCE
# ============================================================
raw_files = collect_files(
    RAW_ROOT,
    ["**/*.csv", "**/*.json", "**/*.parquet", "**/*.txt", "**/*.tsv", "**/*.xlsx", "**/*.xls"]
)

storage_scripts_configs = find_storage_scripts_and_configs()
logs_reports = find_logs_and_reports()

latest_log_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt", "**/*.log", "**/*.txt"])
latest_report_json = latest_file(REPORTS_DIR, ["**/*report*.json"])
latest_report_pdf = latest_file(REPORTS_DIR, ["**/*report*.pdf"])

storage_observations = detect_storage_observations(raw_files)
raw_tree = build_tree_text(RAW_ROOT, max_depth=6, max_items=300)
latest_log_preview = read_text_preview(latest_log_txt, max_lines=35) if latest_log_txt else "No text log found."
latest_json_preview = read_text_preview(latest_report_json, max_lines=35) if latest_report_json else "No JSON report found."

print("\nStorage observations:")
for item in storage_observations:
    print("-", item)

print("\nDiscovered raw files:", len(raw_files))
print("Discovered scripts/configs:", len(storage_scripts_configs))
print("Discovered logs/reports:", len(logs_reports))

# ============================================================
# 5) BUILD DATA TABLES
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

raw_file_rows = [["Source", "File Name", "Relative Path", "Type", "Last Modified"]]
for p in raw_files[:50]:
    info = file_info(p)
    rel = p.relative_to(RAW_ROOT) if RAW_ROOT.exists() else p
    source = rel.parts[0] if len(rel.parts) > 0 else "Unknown"
    raw_file_rows.append([
        source,
        info["name"],
        info["relative_path"],
        info["suffix"] or "N/A",
        info["modified"],
    ])
if len(raw_file_rows) == 1:
    raw_file_rows.append(["No raw files found", "-", "-", "-", "-"])

script_config_rows = [["Script / Config", "Relative Path", "Type", "Last Modified", "Size"]]
for p in storage_scripts_configs[:25]:
    info = file_info(p)
    file_type = "Notebook" if p.suffix.lower() == ".ipynb" else ("Python Script" if p.suffix.lower() == ".py" else "Config")
    script_config_rows.append([
        info["name"],
        info["relative_path"],
        file_type,
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(script_config_rows) == 1:
    script_config_rows.append(["No script/config found", "-", "-", "-", "-"])

log_rows = [["Log / Report File", "Relative Path", "Last Modified", "Size"]]
for p in logs_reports[:25]:
    info = file_info(p)
    log_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(log_rows) == 1:
    log_rows.append(["No logs/reports found", "-", "-", "-"])

team_table = make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch])

raw_file_table = make_wrapped_table(
    raw_file_rows,
    col_widths=[1.1 * inch, 1.5 * inch, 2.55 * inch, 0.6 * inch, 1.0 * inch],
    path_cols=[2],
    file_cols=[1]
)

script_config_table = make_wrapped_table(
    script_config_rows,
    col_widths=[1.55 * inch, 2.95 * inch, 0.85 * inch, 1.0 * inch, 0.65 * inch],
    path_cols=[1],
    file_cols=[0]
)

log_table = make_wrapped_table(
    log_rows,
    col_widths=[1.8 * inch, 3.05 * inch, 0.95 * inch, 0.65 * inch],
    path_cols=[1],
    file_cols=[0]
)

# ============================================================
# 6) PREVIEW CONTENT
# ============================================================
preview_items = []
for p in storage_scripts_configs[:5]:
    preview_items.append({
        "name": p.name,
        "relative_path": rel_path(p),
        "preview": read_text_preview(p, max_lines=40, max_chars=3200),
    })

# ============================================================
# 7) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("03 Raw Data Storage", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the raw data storage layer for the recommendation pipeline. It captures evidence from the project structure and available logs to show how ingested data is stored, how the folder layout is organized, and which scripts or configuration files support the storage process.",
    body_style
))

story.append(Paragraph("2. Raw Data Storage Objective Coverage", heading_style))
story.append(Paragraph("• Store ingested data in a local data lake or cloud-like local filesystem structure.", bullet_style))
story.append(Paragraph("• Use a structured folder layout partitioned by source, type, and timestamp where available.", bullet_style))
story.append(Paragraph("• Provide storage structure documentation.", bullet_style))
story.append(Paragraph("• Provide upload scripts or configuration file evidence.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Raw Data Storage Summary", heading_style))
for item in storage_observations:
    story.append(Paragraph(f"• {escape(item)}", bullet_style))
story.append(Spacer(1, 8))

story.append(Paragraph("4. Raw Data Files Identified", heading_style))
story.append(Paragraph(
    "The table below lists discovered raw data assets under the local raw storage layer. Long file names and paths are wrapped explicitly so they remain readable in the PDF.",
    body_style
))
story.append(raw_file_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Storage Structure Documentation", heading_style))
story.append(Paragraph(
    "The raw storage tree below documents the folder layout used for ingested files. This provides evidence of source-based organization and timestamp-based partitions where those folders exist.",
    body_style
))
story.append(Preformatted(wrap_block_text(raw_tree, width=92), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("6. Upload Scripts or Configuration Files", heading_style))
story.append(Paragraph(
    "The following scripts, notebooks, and configuration files were identified as likely storage, upload, ingestion, or load-related assets in the project.",
    body_style
))
story.append(script_config_table)
story.append(Spacer(1, 12))

for item in preview_items:
    story.append(Paragraph(f"Preview: {escape(item['name'])}", sub_heading_style))
    story.append(Paragraph(f"<b>Path:</b> {escape(item['relative_path'])}", meta_style))
    story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
    story.append(Spacer(1, 8))

story.append(PageBreak())

story.append(Paragraph("7. Logs and Reports Related to Storage", heading_style))
story.append(Paragraph(
    "The table below lists discovered logs and generated reports from the project. Wrapped table cells are used so long report names and relative paths do not get cut off.",
    body_style
))
story.append(log_table)
story.append(Spacer(1, 12))

story.append(Paragraph("8. Latest Log Preview", heading_style))
story.append(Paragraph(
    "Long log lines are manually wrapped before being sent to the PDF preformatted block so that the output remains presentable.",
    body_style
))
story.append(Preformatted(wrap_block_text(latest_log_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("9. Latest JSON Report Preview", heading_style))
story.append(Preformatted(wrap_block_text(latest_json_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("10. Conclusion", heading_style))
story.append(Paragraph(
    "Based on the discovered project artifacts, the raw data storage layer is documented through the raw folder structure, identified raw files, related scripts or configuration files, and supporting logs or reports. This PDF provides submission-ready evidence for the Raw Data Storage deliverable.",
    body_style
))

# ============================================================
# 8) BUILD PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"03 Raw Data Storage- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal PDF was open or locked.")
    print(f"Saved alternate file instead: {alt_path}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
REPORTS_DIR: C:\Users\barath\recomart-pipeline\reports
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\03 Raw Data Storage- DM4ML-Group51.pdf

Storage observations:
- Raw files discovered: 6
- Sources identified: dummyjson, retailrocket
- Timestamp partition load_date found: Yes
- Timestamp partition load_hour found: Yes
- File types found: .csv, .json

Discovered raw files: 6
Discovered scripts/configs: 223
Discovered logs/reports: 102

PDF created successfully: C:\Users\barath\recomart-pipeline\03 Raw Data Storage- DM4ML-Group51.pdf
